# 02 — Statistical Analysis
This notebook demonstrates descriptive statistical analysis and simple relationship analysis using the e-commerce dataset.

**Concepts demonstrated:** descriptive statistics, mean, median, standard deviation, quartiles, correlation, group summaries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load and prepare data

In [ ]:
RAW = Path('../data/raw')
orders = pd.read_csv(RAW / 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(RAW / 'order_items.csv')
marketing = pd.read_csv(RAW / 'marketing_spend.csv', parse_dates=['spend_date'])

df = orders.merge(order_items, on='order_id', how='inner')
df['revenue'] = df['quantity'] * df['item_price']
df.head()

## 2. Descriptive statistics for revenue

In [ ]:
print(df['revenue'].describe())
print('Median revenue:', df['revenue'].median())
print('Standard deviation:', df['revenue'].std())

## 3. Customer spend distribution

In [ ]:
customer_spend = df.groupby('customer_id')['revenue'].sum().reset_index(name='total_spend')
print(customer_spend['total_spend'].describe())
customer_spend['spend_quartile'] = pd.qcut(customer_spend['total_spend'], 4, labels=['Q1','Q2','Q3','Q4'])
customer_spend.head()

## 4. Product-level summary statistics

In [ ]:
product_stats = df.groupby('product_id')['revenue'].agg(
    mean_revenue='mean',
    median_revenue='median',
    std_revenue='std',
    min_revenue='min',
    max_revenue='max'
).reset_index()
product_stats.head()

## 5. Correlation between marketing spend and revenue
This is a simple example of relationship analysis.

In [ ]:
daily_rev = df.groupby(df['order_date'].dt.date)['revenue'].sum().reset_index()
daily_rev.columns = ['date', 'daily_revenue']
marketing_daily = marketing.groupby('spend_date')['spend'].sum().reset_index()
marketing_daily.columns = ['date', 'daily_spend']
marketing_daily['date'] = marketing_daily['date'].dt.date

corr_df = marketing_daily.merge(daily_rev, on='date', how='inner')
corr_df[['daily_spend', 'daily_revenue']].corr()

## 6. Key business interpretation ideas
- Is revenue skewed by a few large orders?
- Which customers fall into the top quartile of spend?
- Does marketing spend move in the same direction as revenue?

## Try it yourself

### Questions
1. Calculate average order value per product.
2. Compare mean vs median customer spend.
3. Check if one marketing channel dominates daily spend.

### Answers

In [ ]:
# 1. Average order value per product
product_aov = df.groupby('product_id').agg(
    total_revenue=('revenue', 'sum'),
    orders=('order_id', 'nunique')
).reset_index()
product_aov['avg_order_value_per_product'] = product_aov['total_revenue'] / product_aov['orders']
display(product_aov.head())

# 2. Mean vs median customer spend
print('Mean customer spend:', customer_spend['total_spend'].mean())
print('Median customer spend:', customer_spend['total_spend'].median())

# 3. Dominant marketing channel by total spend
channel_spend = marketing.groupby('channel')['spend'].sum().sort_values(ascending=False)
print(channel_spend)